# HOWTO: Profiling Python Code
by [Michael Hahsler](https://michael.hahsler.net)

It is often important to compare the run-time of different algorithms or to find out where an implementation wastes time or consumes too much memory. 
Details on timing code and memory profiling in notebooks can be found in in
[Python Data Science Handbook: Profiling](https://jakevdp.github.io/PythonDataScienceHandbook/01.07-timing-and-profiling.html) by Jake VanderPlas.

The best option is to use the dedicated cProfile tool. In some cases, more lightweight timing is
useful. We will discuss this further down 


## Profile Timing with cProfile

Profiling timing can be used to find the part of your program that uses up most of the time.
See: [Python Docs: The Python Profilers](https://docs.python.org/3/library/profile.html)

In Jupyter notebooks, we can use the convenient magic command `%prun`. `%prun -l 10` shows only the 10
most time-consuming functions.

In [9]:
def my_function(n = 1000):
    for i in range(n):
        sum(range(n))

%prun -l 10 my_function()

         1382 function calls (1374 primitive calls) in 0.023 seconds

   Ordered by: internal time
   List reduced from 129 to 10 due to restriction <10>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
     1000    0.012    0.000    0.012    0.000 {built-in method builtins.sum}
      5/2    0.005    0.001    0.005    0.003 {method 'run' of '_contextvars.Context' objects}
      2/1    0.003    0.002    0.004    0.004 <string>:1(<module>)
        1    0.001    0.001    0.005    0.005 zmqstream.py:546(_run_callback)
        6    0.000    0.000    0.000    0.000 socket.py:623(send)
        1    0.000    0.000    0.004    0.004 2873108602.py:1(my_function)
        1    0.000    0.000    0.005    0.005 history.py:1008(_writeout_input_cache)
        2    0.000    0.000    0.000    0.000 {method 'recv' of '_socket.socket' objects}
        1    0.000    0.000    0.000    0.000 {method 'disable' of '_lsprof.Profiler' objects}
        1    0.000    0.000    0.000    0.000

We see that most time (see `tottime`) is spent on the 1000 calls to `builtins.sum`. Making this part of the code faster is what you should be focusing on.

Many functions are shown that come not from your code, but from the Jupyer infrastructure. Ignore them.

The profiler can also be called in regular Python code.

In [ ]:
import cProfile

def my_function(n = 1000):
    for i in range(n):
        sum(range(n))

cProfile.run('my_function()', sort='time')

         1126 function calls (1124 primitive calls) in 0.011 seconds

   Ordered by: internal time

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
     1000    0.007    0.000    0.007    0.000 {built-in method builtins.sum}
      2/1    0.004    0.002    0.003    0.003 <string>:1(<module>)
        1    0.000    0.000    0.003    0.003 kernelbase.py:324(_flush)
        1    0.000    0.000    0.003    0.003 2216933895.py:3(my_function)
        1    0.000    0.000    0.000    0.000 {method 'disable' of '_lsprof.Profiler' objects}
      2/1    0.000    0.000    0.003    0.003 {built-in method builtins.exec}
        1    0.000    0.000    0.000    0.000 inspect.py:3102(_bind)
        1    0.000    0.000    0.000    0.000 inspect.py:2888(kwargs)
        1    0.000    0.000    0.000    0.000 enum.py:1562(__and__)
        1    0.000    0.000    0.003    0.003 history.py:92(only_when_enabled)
        1    0.000    0.000    0.000    0.000 zmqstream.py:458(update_flag)
  

## Profile Memory Usage

Some ML and AI algorithms tend to run out of memory. The first thing you should do is to run on Linux-based systems `top` in a shell and find the process that runs your code. On Windows you can run the Task Manager to monitor memory usage.

Memory profiling in Python is a little more involved. Read Profiling Memory Use in
[Python Data Science Handbook: Profiling](https://jakevdp.github.io/PythonDataScienceHandbook/01.07-timing-and-profiling.html).

## Manual timing using the time and timeit packages

In Jupyter notebooks you can use the magic time command to time individual lines.

In [7]:
%time for i in range(100000): pass

CPU times: user 1.81 ms, sys: 35 μs, total: 1.84 ms
Wall time: 1.69 ms


The `timit` package is useful to measure time for code that is called repeatedly.

In [8]:
%timeit for i in range(100000): pass

1.41 ms ± 76.2 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


The code was run 100 times and the average time and standard deviation are reported.
Note that repeatedly executing the same code is much faster in Python since most of the interpreter work is done only during the first time the code is executed.